**Task 4:** Write a function that takes as input two timestamps of the form 2017/05/13 12:00 and calculates their differences in hours. Only return the full hour difference, rounded. E.g. 2022/02/15 00:05 and 2022/02/15 01:00 would return 1 hour.

**Task 5:** Expand the above function to only count the time difference between 09:00–17:00 and only on weekdays.

Single pandas-based implementation for both: `hours_between(time, business_hours_only=False)`.

In [7]:
import datetime
import pandas as pd

In [64]:
def hours_between(time, business_hours_only=False):
    start, end = time
    fmt = "%Y/%m/%d %H:%M"
    t1 = pd.Timestamp(datetime.datetime.strptime(start, fmt))
    t2 = pd.Timestamp(datetime.datetime.strptime(end, fmt))

    if t1 > t2:
        t1, t2 = t2, t1

    if not business_hours_only:
        return round((t2 - t1).total_seconds() / 3600)

    # Task 5: only count minutes on weekdays (Mon-Fri) between 09:00 and 17:00
    minutes = pd.date_range(start=t1, end=t2, freq="min", inclusive="left")
    business_minutes = minutes[
        (minutes.weekday < 5)
        & (minutes.time >= datetime.time(9, 0))
        & (minutes.time < datetime.time(17, 0))
    ]
    return round(len(business_minutes) / 60)

### Task 4 test cases (default, `business_hours_only=False`)

In [58]:
test_cases = [
    ("2022/02/15 00:05", "2022/02/15 01:00"),
    ("2017/05/13 12:00", "2017/05/13 12:00"),
    ("2017/05/13 12:00", "2017/05/13 12:29"),
    ("2017/05/13 12:00", "2017/05/13 12:30"),
    ("2017/05/13 12:00", "2017/05/13 12:31"),
    ("2017/05/13 12:00", "2017/05/14 12:00"),
    ("2017/05/13 12:00", "2017/05/13 09:00"),
    ("2017/01/31 23:30", "2017/02/01 00:00"),
    ("2017/12/31 23:00", "2018/01/01 01:00"),
    ("2020/02/28 23:00", "2020/03/01 01:00"),
    ("2021/02/28 23:00", "2021/03/01 01:00"),
    ("2017/05/13 00:00", "2019/05/13 00:00"),
]

for period in test_cases:
    print(f"{period[0]} -> {period[1]} = {hours_between(period)}h")

2022/02/15 00:05 -> 2022/02/15 01:00 = 1h
2017/05/13 12:00 -> 2017/05/13 12:00 = 0h
2017/05/13 12:00 -> 2017/05/13 12:29 = 0h
2017/05/13 12:00 -> 2017/05/13 12:30 = 0h
2017/05/13 12:00 -> 2017/05/13 12:31 = 1h
2017/05/13 12:00 -> 2017/05/14 12:00 = 24h
2017/05/13 12:00 -> 2017/05/13 09:00 = 3h
2017/01/31 23:30 -> 2017/02/01 00:00 = 0h
2017/12/31 23:00 -> 2018/01/01 01:00 = 2h
2020/02/28 23:00 -> 2020/03/01 01:00 = 26h
2021/02/28 23:00 -> 2021/03/01 01:00 = 2h
2017/05/13 00:00 -> 2019/05/13 00:00 = 17520h


### Task 5 test cases (`business_hours_only=True`)

In [69]:
business_test_cases = [
    ("2017/05/15 09:00", "2017/05/15 10:00"),  # Monday, fully inside hours -> 1h
    ("2017/05/15 08:00", "2017/05/15 10:00"),  # starts before 09:00, clipped -> 1h
    ("2017/05/15 16:00", "2017/05/15 18:00"),  # ends after 17:00, clipped -> 1h
    ("2017/05/13 09:00", "2017/05/13 17:00"),  # Saturday -> 0h
    ("2017/05/12 16:00", "2017/05/15 10:00"),  # Fri 16:00 -> Mon 10:00, spans weekend -> 2h
    ("2017/05/15 09:00", "2017/05/19 17:00"),  # full Mon-Fri work week -> 40h
    ("2017/05/15 12:00", "2017/05/15 12:00"),  # identical timestamps -> 0h
]

for period in business_test_cases:
    print(f"{period[0]} -> {period[1]} = {hours_between(period, business_hours_only=True)}h")

2017/05/15 09:00 -> 2017/05/15 10:00 = 1h
2017/05/15 08:00 -> 2017/05/15 10:00 = 1h
2017/05/15 16:00 -> 2017/05/15 18:00 = 1h
2017/05/13 09:00 -> 2017/05/13 17:00 = 0h
2017/05/12 16:00 -> 2017/05/15 10:00 = 2h
2017/05/15 09:00 -> 2017/05/19 17:00 = 40h
2017/05/15 12:00 -> 2017/05/15 12:00 = 0h
